# P05 数据处理：Pandas 表格操作（最重要！）


## 本节你能学到什么
- 读取 CSV / Excel 文件
- 查看、筛选、修改数据
- 处理缺失值
- 分组统计 `groupby`
- **经济学练习**：A 股财务数据分析，计算各年度平均 ROE


---
## 1. 基础：DataFrame 是什么？
DataFrame 就是一张「带列名的 Excel 表格」，行 = 样本，列 = 变量。


In [ ]:
import pandas as pd
import numpy as np

# 手动创建一个小 DataFrame
df = pd.DataFrame({
    '股票代码': ['000001', '000002', '600036', '600519'],
    '公司名称': ['平安银行', '万科A', '招商银行', '贵州茅台'],
    '年份':     [2022, 2022, 2022, 2022],
    'ROE':      [0.112, 0.089, 0.164, 0.305],
    '总资产(亿)': [5800, 1200, 10200, 1650],
})
df


---
## 2. 查看数据


In [ ]:
# 构造更大的示例数据
np.random.seed(0)
n = 200
industries = ['制造业','金融业','零售业','房地产']
big = pd.DataFrame({
    '股票代码': [f'{i:06d}' for i in range(1, n+1)],
    '行业':     np.random.choice(industries, n),
    '年份':     np.random.choice([2020,2021,2022], n),
    'ROE':      np.round(np.random.normal(0.1, 0.08, n), 4),
    '总资产(亿)': np.round(np.random.lognormal(6, 1.5, n), 1),
})

print('形状（行×列）:', big.shape)
big.head(5)  # 前5行


In [ ]:
big.info()      # 列名、非空数量、类型


In [ ]:
big.describe()  # 数值列描述统计


---
## 3. 选取列与行


In [ ]:
# 选单列 → Series
print(big['ROE'].head())

# 选多列 → DataFrame
sub = big[['股票代码','行业','ROE']]
sub.head(3)


In [ ]:
# loc：按标签/条件定位
print(big.loc[0, 'ROE'])       # 第0行 ROE 列
print(big.loc[0:2, 'ROE':'年份'])  # 行0-2，列 ROE 到 年份

# iloc：按位置定位
print(big.iloc[0, 3])          # 第0行第3列
print(big.iloc[:3, :3])        # 前3行前3列


---
## 4. 筛选数据


In [ ]:
# 单条件筛选：ROE > 0.15
high_roe = big[big['ROE'] > 0.15]
print(f'高ROE企业数量: {len(high_roe)}')
high_roe.head(3)


In [ ]:
# 多条件筛选（& 且，| 或，~ 非）
mfg_high = big[(big['行业'] == '制造业') & (big['ROE'] > 0.1)]
print(f'制造业且ROE>10%的企业: {len(mfg_high)}')

# isin() 筛选多个值
fin_re = big[big['行业'].isin(['金融业', '房地产'])]
print(f'金融或房地产企业: {len(fin_re)}')


---
## 5. 新增列、删除列


In [ ]:
# 新增衍生变量
big['log_资产'] = np.log(big['总资产(亿)'])
big['ROE_pct']  = (big['ROE'] * 100).round(2)  # 转为百分比

# 删除列
big2 = big.drop(columns=['ROE_pct'])
print('删除前列数:', big.shape[1], '删除后列数:', big2.shape[1])
big.head(3)


---
## 6. 处理缺失值


In [ ]:
# 人为制造一些缺失值
df_miss = big.copy()
df_miss.loc[np.random.choice(n, 20), 'ROE'] = np.nan

print('缺失值数量:\n', df_miss.isnull().sum())

# 查看有缺失的行
df_miss[df_miss['ROE'].isnull()].head(3)


In [ ]:
# 策略1：删掉有缺失的行
df_drop = df_miss.dropna(subset=['ROE'])
print('删除后行数:', len(df_drop))

# 策略2：用均值填充
df_fill = df_miss.copy()
df_fill['ROE'] = df_fill['ROE'].fillna(df_fill['ROE'].mean())
print('填充后缺失数:', df_fill['ROE'].isnull().sum())


---
## 7. 分组统计 groupby


In [ ]:
# 按行业分组，计算各行业ROE均值、标准差、企业数
grouped = big.groupby('行业')['ROE'].agg(['mean','std','count'])
grouped.columns = ['平均ROE','ROE波动','企业数']
grouped.round(4)


In [ ]:
# 多字段分组：行业 × 年份
year_ind = big.groupby(['年份','行业'])['ROE'].mean().unstack()
year_ind.round(4)


---
## 8. 保存数据


In [ ]:
# 保存为 CSV
# big.to_csv('output.csv', index=False, encoding='utf-8-sig')  # utf-8-sig 避免Excel打开乱码

# 保存为 Excel
# big.to_excel('output.xlsx', index=False)

print('（取消注释即可运行保存）')


---
## 9. 经济学场景练习：A 股财务数据分析


In [ ]:
# 完整练习：从生成数据到分析全流程
np.random.seed(99)
N = 500
fin_data = pd.DataFrame({
    'stkcd':   [f'{i:06d}' for i in range(1, N+1)],
    'year':    np.random.choice([2018,2019,2020,2021,2022], N),
    'ind':     np.random.choice(['制造业','金融业','零售业','房地产','信息技术'], N),
    'roe':     np.round(np.random.normal(0.09, 0.1, N), 4),
    'lev':     np.round(np.random.uniform(0.2, 0.8, N), 4),  # 资产负债率
    'size':    np.round(np.random.lognormal(5, 1.2, N), 2),   # 总资产（亿）
})

# 1. 基本情况
print('样本量:', len(fin_data))
print('年份范围:', fin_data['year'].min(), '-', fin_data['year'].max())
fin_data.head(3)


In [ ]:
# 2. 筛选制造业
mfg = fin_data[fin_data['ind'] == '制造业'].copy()
print(f'制造业样本: {len(mfg)} 条')

# 3. 各年度平均 ROE
roe_by_year = mfg.groupby('year')['roe'].agg(['mean','count'])
roe_by_year.columns = ['平均ROE','样本数']
print('\n制造业各年度平均ROE:')
print(roe_by_year.round(4))


In [ ]:
# 4. 简单描述性统计
print('制造业核心指标描述统计:')
mfg[['roe','lev','size']].describe().round(4)


---
## 练习题
1. 从 `fin_data` 中筛选出 `lev > 0.6`（高杠杆）且 `roe > 0.1` 的企业，统计各行业数量。
2. 计算每年、每个行业的平均资产负债率（`lev`），找出 2022 年杠杆最高的行业。
3. 新建一列 `log_size = log(size)`，然后按行业计算 `log_size` 的均值，从大到小排序。
   - 提示：`groupby(...).mean().sort_values(ascending=False)`
